In [22]:
using LowLevelFEM, LinearAlgebra

In [23]:
Threads.nthreads()
LinearAlgebra.BLAS.get_num_threads()

2

In [24]:
structured_box_mesh(n=10, order=2)

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=3, field=:u)

Problem("structured_box", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 9261, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs)

In [25]:
prob = Problem([mat])
@time K1 = stiffnessMatrix(prob)

  3.105439 seconds (4.92 M allocations: 6.271 GiB, 30.06% gc time)


sparse([1, 2, 3, 49, 50, 51, 79, 80, 81, 82  …  27711, 27721, 27722, 27723, 27775, 27776, 27777, 27781, 27782, 27783], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783], [1754.985754985726, 641.0256410256294, -641.0256410256311, -313.39031339031175, -160.25641025640905, -42.73504273503918, 353.27635327636517, 320.5128205128236, 170.94017094015834, -313.390313390307  …  -820.5128205128224, 4.774847184307873e-12, -2.6542323894318542e-11, 729.3447293447002, -3.1889157980913296e-11, 8.494538406011998e-12, 729.344729344696, 1.0530243343964685e-11, 5.667288860422559e-11, 64182.33618233609], 27783, 27783)

In [26]:
μ = mat.μ
λ = mat.λ
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

6×6 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0      0.0      0.0
 0.0        0.0        0.0        76923.1      0.0      0.0
 0.0        0.0        0.0            0.0  76923.1      0.0
 0.0        0.0        0.0            0.0      0.0  76923.1

In [27]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:csc, threads=1);

  0.718683 seconds (85.61 k allocations: 104.101 MiB)


In [28]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:csc);

  0.424580 seconds (85.84 k allocations: 213.902 MiB)


In [29]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv, threads=1);

  1.213714 seconds (95.81 k allocations: 1.495 GiB, 18.63% gc time)


In [30]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv, threads=2);

  1.016891 seconds (95.89 k allocations: 1.495 GiB, 32.86% gc time)


In [31]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv);

  0.955874 seconds (96.21 k allocations: 1.495 GiB, 22.79% gc time)


In [32]:
norm(K1.A - K2.A) / norm(K1.A)

2.3907800717529193e-16

In [33]:
using Profile

Kpattern = build_csc_pattern(Pu, Pu; Ω="body")

# Compilation
fill!(Kpattern.nzval, 0.0)
∫(
    SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu);
    Ω="body",
    assembly=:csc,
    threads=1,
    csc_matrix=Kpattern
)

fill!(Kpattern.nzval, 0.0)
Profile.Allocs.clear()

Profile.Allocs.@profile sample_rate=0.0001 begin
    ∫(
        SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu);
        Ω="body",
        assembly=:csc,
        threads=1,
        csc_matrix=Kpattern
    )
end

prof = Profile.Allocs.fetch()
length(prof.allocs)

0

In [34]:
structured_rect_mesh(x0=10.0, n=50, order=2)

In [35]:
prob = Problem([mat], type=:AxiSymmetric)

@time K1 = stiffnessMatrix(prob)

  0.153943 seconds (1.11 M allocations: 275.582 MiB)


sparse([1, 2, 9, 10, 107, 108, 699, 700, 799, 800  …  503, 504, 5601, 5602, 20201, 20202, 20397, 20398, 20401, 20402], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402], [6.767206588220727e6, 3.0206010595967714e6, 376292.68629678735, -201545.25177670617, -5.265845710336073e6, 805858.7924758468, -1.1009271415975492e6, 201115.6322682682, 912673.5440149194, -804462.5290736933  …  -4296.195080689155, -2.455189565365407e7, -5.898675847354304e6, -4.2459295993711185e6, 8.323695510625839e-8, -946881.3960378015, 4296.195080269768, -2.455189565365311e7, -2.8032809495925903e-7, 6.798986488703498e7], 20402, 20402)

In [36]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u)

E = mat.E
ν = mat.ν

r = ScalarField(Pu, "body", (x, y, z)->x)
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]
A2 = [0 0; 1/r 0; 0 0; 0 0]
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu
D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

In [44]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:csc, threads=1);

  0.721300 seconds (3.18 M allocations: 207.538 MiB, 23.17% gc time)


In [46]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:csc);

  0.323214 seconds (3.18 M allocations: 222.397 MiB, 8.13% gc time)


In [47]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv, threads=1);

  0.765345 seconds (3.52 M allocations: 302.928 MiB, 27.27% gc time)


In [53]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv, threads=2);

  1.079374 seconds (3.52 M allocations: 302.975 MiB, 31.96% gc time)


In [50]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv);

  0.884435 seconds (3.52 M allocations: 303.249 MiB, 31.85% gc time)


In [42]:
norm(K1.A - K2.A) / norm(K1.A)

3.6542243096957084e-14